**Abby Ortego**

**CMPS 6790 Data Science**

**Milestone 1**

# Imports

In [13]:
import requests
from bs4 import BeautifulSoup
import regex as re
import pandas as pd

# Motivation

[Magic: The Gathering](https://magic.wizards.com/en/intro) is a popular trading card game with competitive tournaments hosted around the globe. During the tournaments, players create their own decks (possibly inspired by existing ones) and battle one another until one lone victor remains. 

This yields the question: what makes a winning deck? 

The community is divided by this. Some would (naively) say it's the price of the cards in the deck while others would argue properties on the cards hold more sway. This notebook is the first step into answering this question with methods grounded in statistical reasoning. 

# Extracting, Transforming, and Loading the Data

To answer this question, we need access to tournament data from Magic: The Gathering events. If we can collect the decks that were used in the tournaments and how they fared, we can start making informed guesses. 

Thankfully, [MTG Top 8](https://mtgtop8.com/) compiles this data from reliable tournament reporting services (e.g., [Melee](https://melee.gg/), [Magic Events](https://magic.gg/), etc.). This site does not offer an API to collect this data, though, so we will have to scrape their pages for the data we're interested in. The site's [policies](https://mtgtop8.com/privacy) do not ban web scrapers. 

## Getting Tournament Data

MTG Top 8 organizes decks by tournament, so we will first need to compile a list of tournaments and their links on the site. Magic: The Gathering is quite popular, though, with a variety of play formats and tournaments hosted all around the globe, so grabbing data for *all* tournaments on the site is not feasible. 

For this notebook, we will focus only on major tournaments played in the Standard format during the last month. [Explain why you chose this scope].

This [link](https://mtgtop8.com/format?f=ST&meta=46&a=) lists all major tournaments in the last two months in the "Last 20 Events" table. This is close to what we're interested in! 

This table only displays 20 items at a time, though, which means we'll have to loop over the pages and scrape this table for each one.

In [3]:
# get all tournaments tables
tournaments_tables = []
for page in range(1, 7):    # there are roughly 6 pages worth of data
    r = requests.get(f"https://mtgtop8.com/format?f=ST&meta=46&cp={page}")
    if r.status_code != 200:
        print(f"Could not scrape events from page {page}.")
        continue
    #

    soup = BeautifulSoup(r.content)
    tables = soup.find_all("table")     # using soup to grab all tables on the page
    tournaments_tables.append(tables[2])   # the table we're interested in is the third one on the page 
#
display(tournaments_tables)

[<table align="center" border="0" class="Stable" width="98%">
 <tr class="hover_tr">
 <td align="center" width="5%"><img height="14" src="/graph/online/mtgo.png" title="MTG Online"/></td>
 <td class="S14" width="70%"><a href="event?e=80490&amp;f=ST">MTGO Challenge 32</a> <span class="new">NEW</span></td>
 <td align="center" width="13%"><img src="/graph/star.png"/><img src="/graph/star.png"/></td>
 <td align="right" class="S12" width="12%">15/02/26</td>
 </tr>
 <tr class="hover_tr">
 <td align="center" width="5%"><img height="17" src="/graph/online/paper.png" title="Paper"/></td>
 <td class="S14" width="70%"><a href="event?e=80514&amp;f=ST">South America Regional Championship Santiago 2026 - Devir </a> @ <a class="und" href="event?e=80514&amp;f=ST">Santiago (Chile)</a> <span class="new">NEW</span></td>
 <td align="center" width="13%"><img src="/graph/star.png"/><img src="/graph/star.png"/><img src="/graph/star.png"/></td>
 <td align="right" class="S12" width="12%">14/02/26</td>
 </tr>
 

Now that we have the tournament tables, we can extract the information we're interested in from the html. We're looking for...
- *Tournament Name* to help us identify tournaments 
- *Tournament Date* to help us only grab tournaments in the last month
- *Link to Additional Tournament Information* to help us grab ranking information later

**(1) Tournament Names and Dates** are stored as text within the html tags, so we can use BeautifulSoup's `get_text()` method to extract this information from the table. When you do you get a string that looks like this:

`'\n\n\nRCQ @ Draco Hobby Center (Bogota, Colombia)\n\n01/02/26\n\n\n\nMTGO RC Super Qualifier\n\n01/02/26\n\n\n\n2nd Chance PTQ @ Pro Tour Lorwyn Eclipsed (Richmond, VA)\n\n31/01/26\n\n\n\n`

It's not very pretty but the new line characters do have a pattern! 3-4 new line characters separate tournaments and 2 new line characters separate tournament name from date. We can use regex to split the string on every 3-4 or 2 new line characters. 

This yields a list where every even indexed item is a tournament name and every odd indexed item is a tournament date. They can be split using list slicing to get names and dates separately while still respecting order so that no information gets lost or confused. 

**(2) Links to Additional Tournament Information** are stored in `a` tags with an `href` property. Some entries in the table record an additional `a` tag with a the `class` property specified in addition to `href`. These tags have duplicate information that we don't need. We can use BeautifulSoup's `find_all()` method to find all `a` tags that have an `href` property specified but no `class` property.

Lastly, we can use the `get()` method to extract the links. 

Below loops through the html for the tournament tables and performs the actions outlined in **(1)** and **(2)** above.

In [38]:
# lists to store this information
names = []
dates = []
links = []

for tournaments_table in tournaments_tables:
    # (1) Tournament Names and Date
    table_text = tournaments_table.get_text()
    name_dates = re.split("\\n{3,4}|\\n{2}", table_text)[1:-1]   # the first and last item are blank from splitting 
    names.extend(name_dates[::2])     # gets all even indexed items (start at position 0 with step size 2)
    dates.extend(name_dates[1::2])      # gets all odd indexed items (start at position 1 with step size 2)

    # (2) Links to Additional Tournament Information
    table_hrefs = tournaments_table.find_all("a", href=True, class_=False)
    links.extend([table_href.get("href") for table_href in table_hrefs])
#

display(names, dates, links)

['MTGO Challenge 32 NEW',
 'South America Regional Championship Santiago 2026 - Devir  @ Santiago (Chile) NEW',
 'MTGO Challenge 32 NEW',
 'MTGO Challenge 32 NEW',
 'MTGO Challenge 64 NEW',
 'MTGO Challenge 32',
 'MTGO Challenge 32',
 'MTGO Challenge 32',
 'MTGO Challenge 64',
 'MTGO Challenge 32',
 'RCQ @ Playtime (Merate, Italy)',
 'MTGO Showcase Challenge',
 'MTGO Challenge 32',
 'MTGO Challenge 32',
 'MTGO Challenge 64',
 'MTGO Challenge 32',
 'MTGO Challenge 32',
 'MTGO Challenge 32',
 'MTGO Challenge 64',
 'MTGO Challenge 32',
 'RCQ @ Draco Hobby Center (Bogota, Colombia)',
 'MTGO RC Super Qualifier',
 '2nd Chance PTQ @ Pro Tour Lorwyn Eclipsed (Richmond, VA)',
 'Champions Cup Special Qualifier @ Kawasaki (Japan)',
 'Pro Tour Lorwyn Eclipsed @ Richmond',
 'MTGO Challenge 32',
 'MTGO Challenge 32',
 'MTGO Challenge 32',
 'MTGO Challenge 32',
 'MTGO Challenge 64',
 'Saturday ReCQ #2 @ SCG CON Portland',
 'Super Sunday RCQ #2 @ SCG CON Portland',
 'Super Sunday RCQ #1 @ SCG CON Port

['15/02/26',
 '14/02/26',
 '14/02/26',
 '14/02/26',
 '13/02/26',
 '13/02/26',
 '12/02/26',
 '10/02/26',
 '09/02/26',
 '08/02/26',
 '08/02/26',
 '07/02/26',
 '07/02/26',
 '07/02/26',
 '06/02/26',
 '06/02/26',
 '05/02/26',
 '03/02/26',
 '02/02/26',
 '01/02/26',
 '01/02/26',
 '01/02/26',
 '31/01/26',
 '30/01/26',
 '30/01/26',
 '30/01/26',
 '30/01/26',
 '29/01/26',
 '27/01/26',
 '26/01/26',
 '25/01/26',
 '25/01/26',
 '25/01/26',
 '25/01/26',
 '25/01/26',
 '25/01/26',
 '24/01/26',
 '24/01/26',
 '24/01/26',
 '24/01/26',
 '24/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '23/01/26',
 '22/01/26',
 '20/01/26',
 '19/01/26',
 '18/01/26',
 '17/01/26',
 '17/01/26',
 '16/01/26',
 '16/01/26',
 '15/01/26',
 '13/01/26',
 '12/01/26',
 '11/01/26',
 '11/01/26',
 '11/01/26',
 '11/01/26',
 '11/01/26',
 '11/01/26',

['event?e=80490&f=ST',
 'event?e=80514&f=ST',
 'event?e=80415&f=ST',
 'event?e=80394&f=ST',
 'event?e=80393&f=ST',
 'event?e=80372&f=ST',
 'event?e=80328&f=ST',
 'event?e=80293&f=ST',
 'event?e=80250&f=ST',
 'event?e=80147&f=ST',
 'event?e=80087&f=ST',
 'event?e=80126&f=ST',
 'event?e=80125&f=ST',
 'event?e=80046&f=ST',
 'event?e=80065&f=ST',
 'event?e=80029&f=ST',
 'event?e=79976&f=ST',
 'event?e=79952&f=ST',
 'event?e=79924&f=ST',
 'event?e=79877&f=ST',
 'event?e=79818&f=ST',
 'event?e=79819&f=ST',
 'event?e=79854&f=ST',
 'event?e=79785&f=ST',
 'event?e=79746&f=ST',
 'event?e=79745&f=ST',
 'event?e=79719&f=ST',
 'event?e=79671&f=ST',
 'event?e=79666&f=ST',
 'event?e=79651&f=ST',
 'event?e=79664&f=ST',
 'event?e=79600&f=ST',
 'event?e=79599&f=ST',
 'event?e=79598&f=ST',
 'event?e=79563&f=ST',
 'event?e=79462&f=ST',
 'event?e=79569&f=ST',
 'event?e=79482&f=ST',
 'event?e=79476&f=ST',
 'event?e=79428&f=ST',
 'event?e=79391&f=ST',
 'event?e=79568&f=ST',
 'event?e=79567&f=ST',
 'event?e=7

We can load this data into a panda's dataframe so we can start transforming it. 

In [39]:
all_tournaments_df = pd.DataFrame({'Name': names, 'Date': dates, 'Link': links})
display(all_tournaments_df, all_tournaments_df.dtypes)

,Name,Date,Link
0,MTGO Challenge 32 NEW,15/02/26,event?e=80490&f=ST
1,South America Regional Championship Santiago 2...,14/02/26,event?e=80514&f=ST
2,MTGO Challenge 32 NEW,14/02/26,event?e=80415&f=ST
3,MTGO Challenge 32 NEW,14/02/26,event?e=80394&f=ST
4,MTGO Challenge 64 NEW,13/02/26,event?e=80393&f=ST
...,...,...,...
115,MTGO Challenge 32,25/12/25,event?e=78314&f=ST
116,MTGO Challenge 32,21/12/25,event?e=78269&f=ST
117,"RCQ @ SpellCrafter's Den (Upper Sandusky, OH)",20/12/25,event?e=78209&f=ST
118,"Champions Cup Special Qualifier @ TC (Osaka, J...",20/12/25,event?e=78207&f=ST


Name    str
Date    str
Link    str
dtype: object

Finally, the dataframe above has *all* large tournaments in the last two months, but this wasn't quite what we were interested in so it needs some polishing.

First, some of these tournaments include Magic: The Gathering games played online in the app which has a similar but alternative system for collecting cards and creating decks. Due to this we want to only look at tournaments played outside of the app in live tournaments.

Thankfully, these online tournaments are denoted with "MTGO" so we can filter the `Name` property of the `tournaments_df` to only include tournaments without "MTGO" in the name. 

In [40]:
# all tournaments that are not online (i.e., do not include "MTGO")
not_online = ~all_tournaments_df["Name"].str.contains("MTGO")

Secondly, we are only concerned with tournaments that occurred in the last full month (as of this milestone this is January 2026), so we'll filter tournaments from before or after that time out as well. This will also allow us to convert our `Date` column in the pandas dataframe to the proper type (datetime).

In [41]:
# format to datetime
all_tournaments_df["Date"] = pd.to_datetime(all_tournaments_df["Date"], format="%d/%m/%y")

# all tournaments in january
in_january = (all_tournaments_df["Date"] >= '2026-01-01') & (all_tournaments_df["Date"] < '2026-02-01')

Thirdly, the `Link` contains only the suffix of the URL. We need to append the base URL ("https://mtgtop8.com") to this column for easy access later.

In [42]:
# adding base_url to links
base_url = "https://mtgtop8.com"
all_tournaments_df["Link"] = base_url + '/' + all_tournaments_df["Link"]

When putting it all together you get the following tournament dataframe. 

In [43]:
# applying our filters
tournaments_df = all_tournaments_df[not_online & in_january].reset_index(drop=True)

display(tournaments_df, tournaments_df.dtypes)

,Name,Date,Link
0,2nd Chance PTQ @ Pro Tour Lorwyn Eclipsed (Ric...,2026-01-31,https://mtgtop8.com/event?e=79854&f=ST
1,Champions Cup Special Qualifier @ Kawasaki (Ja...,2026-01-30,https://mtgtop8.com/event?e=79785&f=ST
2,Pro Tour Lorwyn Eclipsed @ Richmond,2026-01-30,https://mtgtop8.com/event?e=79746&f=ST
3,Saturday ReCQ #2 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79664&f=ST
4,Super Sunday RCQ #2 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79600&f=ST
5,Super Sunday RCQ #1 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79599&f=ST
6,Izzet Explosive Experiment Event,2026-01-25,https://mtgtop8.com/event?e=79598&f=ST
7,Regional Championship @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79462&f=ST
8,Saturday RCQ @ SCG CON Portland,2026-01-24,https://mtgtop8.com/event?e=79569&f=ST
9,"RCQ @ BB-Spiele (Rosenheim, Germany)",2026-01-24,https://mtgtop8.com/event?e=79428&f=ST


Name               str
Date    datetime64[us]
Link               str
dtype: object

All in all, we're left with 44 tournaments to grab additional information for (e.g., deck rankings, prices, etc.)

## Getting Additional Tournament Details

# Exploratory Data Analysis

# Future Work